## 1. updated_code_with_random_split

In [1]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.metrics import accuracy_score
import time

# define dataset class
# Wrapping data as PyTorch dataset objects so that they can be used by PyTorch's DataLoader
class BrakeDataset(Dataset):
    # Initialize the dataset, convert the data and labels to PyTorch's Tensor
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    # Returns the length of the dataset
    def __len__(self):
        return len(self.labels)

    # Fetch the corresponding data and label according to the index
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

def load_data(train_path, test_path):
    with open(train_path, 'rb') as f:
        train_data = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_data = pickle.load(f)

    x_train = np.array(train_data['sensor_data'].tolist())
    y_train = np.array(train_data['label'])
    x_test = np.array(test_data['sensor_data'].tolist())
    y_test = np.array(test_data['label'])

    return x_train, y_train, x_test, y_test

class FCN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(FCN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=8)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5)
        self.conv3 = nn.Conv1d(128, 64, kernel_size=3)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = self.global_pool(x).squeeze(-1)
        x = self.fc(x)
        return x

def train_model_multiple_splits(model, dataset, criterion, optimizer, epochs, device, num_splits=5):
    best_val_accuracy = 0.0

    for split_index in range(num_splits):
        print(f"Training with split {split_index + 1}/{num_splits}")

        # Randomly split dataset into training and validation
        train_size = int(0.8 * len(dataset))
        val_size = len(dataset) - train_size
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

        model = model.to(device)
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0

            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                outputs = model(x_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            # Validation
            model.eval()
            val_preds, val_labels = [], []
            with torch.no_grad():
                for x_batch, y_batch in val_loader:
                    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                    outputs = model(x_batch)
                    _, preds = torch.max(outputs, 1)
                    val_preds.extend(preds.cpu().numpy())
                    val_labels.extend(y_batch.cpu().numpy())

            val_accuracy = accuracy_score(val_labels, val_preds)
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss / len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

            # Save best model
            if val_accuracy > best_val_accuracy:
                best_val_accuracy = val_accuracy
                best_model_state = model.state_dict()

    # Load best model state
    model.load_state_dict(best_model_state)
    return model

def test_model(model, test_loader, device):
    model = model.to(device)
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(y_batch.cpu().numpy())

    test_accuracy = accuracy_score(test_labels, test_preds)
    return test_accuracy

if __name__ == "__main__":
    train_path = 'train.pickle'
    test_path = 'test.pickle'

    x_train, y_train, x_test, y_test = load_data(train_path, test_path)
    print(f"Training data shape: {x_train.shape}, Labels shape: {y_train.shape}")
    print(f"Test data shape: {x_test.shape}, Labels shape: {y_test.shape}")

    x_train = np.transpose(x_train, (0, 2, 1))
    x_test = np.transpose(x_test, (0, 2, 1))

    dataset = BrakeDataset(x_train, y_train)
    test_dataset = BrakeDataset(x_test, y_test)

    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FCN(input_channels=x_train.shape[1], num_classes=len(np.unique(y_train)))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    model = train_model_multiple_splits(model, dataset, criterion, optimizer, epochs=20, device=device, num_splits=5)
    test_accuracy = test_model(model, test_loader, device)
    print(f"Test Accuracy: {test_accuracy:.4f}")

Training data shape: (4053, 128, 6), Labels shape: (4053,)
Test data shape: (1001, 128, 6), Labels shape: (1001,)
Training with split 1/5
Epoch 1/20, Loss: 1.8472, Val Accuracy: 0.5672
Epoch 2/20, Loss: 0.8401, Val Accuracy: 0.9001
Epoch 3/20, Loss: 0.2822, Val Accuracy: 0.9642
Epoch 4/20, Loss: 0.1793, Val Accuracy: 0.9593
Epoch 5/20, Loss: 0.1462, Val Accuracy: 0.9803
Epoch 6/20, Loss: 0.0993, Val Accuracy: 0.9815
Epoch 7/20, Loss: 0.0768, Val Accuracy: 0.9852
Epoch 8/20, Loss: 0.0653, Val Accuracy: 0.9778
Epoch 9/20, Loss: 0.0481, Val Accuracy: 0.9889
Epoch 10/20, Loss: 0.0541, Val Accuracy: 0.9914
Epoch 11/20, Loss: 0.0356, Val Accuracy: 0.9901
Epoch 12/20, Loss: 0.0328, Val Accuracy: 0.9889
Epoch 13/20, Loss: 0.0245, Val Accuracy: 0.9901
Epoch 14/20, Loss: 0.0244, Val Accuracy: 0.9914
Epoch 15/20, Loss: 0.0249, Val Accuracy: 0.9938
Epoch 16/20, Loss: 0.0188, Val Accuracy: 0.9938
Epoch 17/20, Loss: 0.0164, Val Accuracy: 0.9889
Epoch 18/20, Loss: 0.0169, Val Accuracy: 0.9938
Epoch 1

## 2. updated_fcn_model: 
### Preprocessed data: Load Data with MaxAbs Scaling
### FCN Model with Dropout and L2 Regularization

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.metrics import accuracy_score
import time

# Define Dataset Class
class BrakeDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Load Data with MaxAbs Scaling
def load_data(train_path, test_path):
    with open(train_path, 'rb') as f:
        train_data = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_data = pickle.load(f)

    x_train = np.array(train_data['sensor_data'].tolist())
    y_train = np.array(train_data['label'])
    x_test = np.array(test_data['sensor_data'].tolist())
    y_test = np.array(test_data['label'])

    # MaxAbs Scaling: 
    # Use maximum absolute value scaling to normalize the data so that the eigenvalues of each sample are scaled to the range [-1, 1].
    x_train = x_train / np.max(np.abs(x_train), axis=1, keepdims=True)
    x_test = x_test / np.max(np.abs(x_test), axis=1, keepdims=True)

    return x_train, y_train, x_test, y_test

# Define FCN Model with Dropout and L2 Regularization
class FCN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(FCN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=8)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5)
        self.conv3 = nn.Conv1d(128, 64, kernel_size=3)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=0.3)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# Train Function
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    model = model.to(device)
    best_val_accuracy = 0.0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                outputs = model(x_batch)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(y_batch.cpu().numpy())

        val_accuracy = accuracy_score(val_labels, val_preds)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss / len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()

    model.load_state_dict(best_model_state)
    return model

# Test Function
def test_model(model, test_loader, device):
    model = model.to(device)
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(y_batch.cpu().numpy())

    test_accuracy = accuracy_score(test_labels, test_preds)
    return test_accuracy

# Main Function
if __name__ == "__main__":
    train_path = 'train.pickle'
    test_path = 'test.pickle'

    x_train, y_train, x_test, y_test = load_data(train_path, test_path)
    print(f"Training data shape: {x_train.shape}, Labels shape: {y_train.shape}")
    print(f"Test data shape: {x_test.shape}, Labels shape: {y_test.shape}")

    x_train = np.transpose(x_train, (0, 2, 1))
    x_test = np.transpose(x_test, (0, 2, 1))

    dataset = BrakeDataset(x_train, y_train)
    test_dataset = BrakeDataset(x_test, y_test)

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FCN(input_channels=x_train.shape[1], num_classes=len(np.unique(y_train)))
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)  # Add L2 regularization

    model = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=30, device=device)
    test_accuracy = test_model(model, test_loader, device)
    print(f"Test Accuracy: {test_accuracy:.4f}")

Training data shape: (4053, 128, 6), Labels shape: (4053,)
Test data shape: (1001, 128, 6), Labels shape: (1001,)
Epoch 1/30, Loss: 1.9107, Val Accuracy: 0.4624
Epoch 2/30, Loss: 1.3220, Val Accuracy: 0.6215
Epoch 3/30, Loss: 0.9557, Val Accuracy: 0.7004
Epoch 4/30, Loss: 0.8198, Val Accuracy: 0.7534
Epoch 5/30, Loss: 0.7206, Val Accuracy: 0.8212
Epoch 6/30, Loss: 0.6621, Val Accuracy: 0.8249
Epoch 7/30, Loss: 0.5820, Val Accuracy: 0.8582
Epoch 8/30, Loss: 0.5209, Val Accuracy: 0.7879
Epoch 9/30, Loss: 0.5009, Val Accuracy: 0.8853
Epoch 10/30, Loss: 0.4569, Val Accuracy: 0.9088
Epoch 11/30, Loss: 0.4289, Val Accuracy: 0.9026
Epoch 12/30, Loss: 0.4241, Val Accuracy: 0.8977
Epoch 13/30, Loss: 0.3948, Val Accuracy: 0.9137
Epoch 14/30, Loss: 0.3775, Val Accuracy: 0.9174
Epoch 15/30, Loss: 0.3666, Val Accuracy: 0.9309
Epoch 16/30, Loss: 0.3645, Val Accuracy: 0.9334
Epoch 17/30, Loss: 0.3565, Val Accuracy: 0.9186
Epoch 18/30, Loss: 0.3518, Val Accuracy: 0.9334
Epoch 19/30, Loss: 0.3472, Val 

## 3. updated_fcn_with_split

In [1]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
import time

# Define Dataset Class
class BrakeDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Load Data
def load_data(train_path, test_path):
    with open(train_path, 'rb') as f:
        train_data = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_data = pickle.load(f)

    x_train = np.array(train_data['sensor_data'].tolist())
    y_train = np.array(train_data['label'])
    x_test = np.array(test_data['sensor_data'].tolist())
    y_test = np.array(test_data['label'])

    return x_train, y_train, x_test, y_test

# Split Data into K-Folds
def create_splits(x_train, y_train, num_splits=5):
    data_size = len(y_train)
    indices = np.arange(data_size)
    np.random.shuffle(indices)
    split_size = data_size // num_splits
    splits = []
    for i in range(num_splits):
        start = i * split_size
        end = start + split_size if i < num_splits - 1 else data_size
        val_indices = indices[start:end]
        train_indices = np.setdiff1d(indices, val_indices)
        splits.append((train_indices, val_indices))
    return splits

# Define FCN Model
class FCN(nn.Module):
    def __init__(self, input_channels, num_classes):
        super(FCN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=8)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5)
        self.conv3 = nn.Conv1d(128, 64, kernel_size=3)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=0.3)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# Train Function
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    model = model.to(device)
    best_val_accuracy = 0.0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                outputs = model(x_batch)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(y_batch.cpu().numpy())

        val_accuracy = accuracy_score(val_labels, val_preds)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss / len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()

    model.load_state_dict(best_model_state)
    return model, best_val_accuracy

# Test Function
def test_model(model, test_loader, device):
    model = model.to(device)
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(y_batch.cpu().numpy())

    test_accuracy = accuracy_score(test_labels, test_preds)
    return test_accuracy

# Main Function
if __name__ == "__main__":
    train_path = 'train.pickle'
    test_path = 'test.pickle'

    x_train, y_train, x_test, y_test = load_data(train_path, test_path)
    print(f"Training data shape: {x_train.shape}, Labels shape: {y_train.shape}")
    print(f"Test data shape: {x_test.shape}, Labels shape: {y_test.shape}")

    x_train = np.transpose(x_train, (0, 2, 1))
    x_test = np.transpose(x_test, (0, 2, 1))

    num_splits = 5
    splits = create_splits(x_train, y_train, num_splits=num_splits)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_channels = x_train.shape[1]
    num_classes = len(np.unique(y_train))

    test_dataset = BrakeDataset(x_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    final_test_accuracy = []

    for split_idx, (train_indices, val_indices) in enumerate(splits):
        print(f"Training with split {split_idx + 1}/{num_splits}")
        x_train_split, y_train_split = x_train[train_indices], y_train[train_indices]
        x_val_split, y_val_split = x_train[val_indices], y_train[val_indices]

        train_dataset = BrakeDataset(x_train_split, y_train_split)
        val_dataset = BrakeDataset(x_val_split, y_val_split)

        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

        model = FCN(input_channels=input_channels, num_classes=num_classes)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.01)

        model, val_accuracy = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20, device=device)
        print(f"Best Validation Accuracy for split {split_idx + 1}: {val_accuracy:.4f}")

        test_accuracy = test_model(model, test_loader, device)
        final_test_accuracy.append(test_accuracy)
        print(f"Test Accuracy for split {split_idx + 1}: {test_accuracy:.4f}")

    avg_test_accuracy = np.mean(final_test_accuracy)
    print(f"Average Test Accuracy across all splits: {avg_test_accuracy:.4f}")

Training data shape: (4053, 128, 6), Labels shape: (4053,)
Test data shape: (1001, 128, 6), Labels shape: (1001,)
Training with split 1/5
Epoch 1/20, Loss: 1.7812, Val Accuracy: 0.6469
Epoch 2/20, Loss: 0.7624, Val Accuracy: 0.9210
Epoch 3/20, Loss: 0.3927, Val Accuracy: 0.9679
Epoch 4/20, Loss: 0.3222, Val Accuracy: 0.9605
Epoch 5/20, Loss: 0.3082, Val Accuracy: 0.9691
Epoch 6/20, Loss: 0.2863, Val Accuracy: 0.9358
Epoch 7/20, Loss: 0.2547, Val Accuracy: 0.9617
Epoch 8/20, Loss: 0.2577, Val Accuracy: 0.9704
Epoch 9/20, Loss: 0.2641, Val Accuracy: 0.9580
Epoch 10/20, Loss: 0.2210, Val Accuracy: 0.9704
Epoch 11/20, Loss: 0.2193, Val Accuracy: 0.9704
Epoch 12/20, Loss: 0.2122, Val Accuracy: 0.9716
Epoch 13/20, Loss: 0.2013, Val Accuracy: 0.9642
Epoch 14/20, Loss: 0.2054, Val Accuracy: 0.9704
Epoch 15/20, Loss: 0.1932, Val Accuracy: 0.9716
Epoch 16/20, Loss: 0.1884, Val Accuracy: 0.9741
Epoch 17/20, Loss: 0.1782, Val Accuracy: 0.9802
Epoch 18/20, Loss: 0.1846, Val Accuracy: 0.9815
Epoch 1

## 4. fcn_advanced_training

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
from collections import Counter
import time
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Dataset Class
class BrakeDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# z-score standardize
def zscore_normalize(train, test):
    # Flatten into two dimensions for easy statistics
    train_2d = train.reshape(-1, train.shape[-1])
    mean = np.mean(train_2d, axis=0)
    std = np.std(train_2d, axis=0)
    train_norm = (train - mean) / (std + 1e-8)
    test_norm = (test - mean) / (std + 1e-8)
    return train_norm, test_norm


# Class balance check
def print_class_distribution(labels, name):
    counter = Counter(labels)
    print(f"{name} class distribution: {dict(counter)}")

# FCN Model
class FCN(nn.Module):
    def __init__(self, input_channels, num_classes, dropout_p=0.5):
        super(FCN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=8)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5)
        self.conv3 = nn.Conv1d(128, 64, kernel_size=3)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# Early Stopping training
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, early_stop_patience=7):
    model = model.to(device)
    best_val_accuracy = 0.0
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                outputs = model(x_batch)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(y_batch.cpu().numpy())

        val_accuracy = accuracy_score(val_labels, val_preds)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss / len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter > early_stop_patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    if best_model_state:
        model.load_state_dict(best_model_state)
    return model, best_val_accuracy

# Test
def test_model(model, test_loader, device):
    model = model.to(device)
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(y_batch.cpu().numpy())
    test_accuracy = accuracy_score(test_labels, test_preds)
    return test_accuracy


if __name__ == "__main__":
    # file path
    train_path = 'train.pickle'
    test_path = 'test.pickle'

    # Load data
    with open(train_path, 'rb') as f:
        train_data = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_data = pickle.load(f)

    x_train = np.array(train_data['sensor_data'].tolist())
    y_train = np.array(train_data['label'])
    x_test = np.array(test_data['sensor_data'].tolist())
    y_test = np.array(test_data['label'])

    print_class_distribution(y_train, "Train")
    print_class_distribution(y_test, "Test")

    # Augmentation
    # x_train, y_train = data_augmentation(x_train, y_train)

    # z-score standardize
    x_train, x_test = zscore_normalize(x_train, x_test)

    # transfer into (batch, channel, timestep)
    x_train = np.transpose(x_train, (0, 2, 1))
    x_test = np.transpose(x_test, (0, 2, 1))

    # Split into training/validation sets
    dataset = BrakeDataset(x_train, y_train)
    test_dataset = BrakeDataset(x_test, y_test)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    # DataLoader
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_channels = x_train.shape[1]
    num_classes = len(np.unique(y_train))
    model = FCN(input_channels=input_channels, num_classes=num_classes, dropout_p=0.5)


    # Compute class weights
    
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    print("Class weights:", class_weights)


    # Loss function and Optimizer
    # criterion = nn.CrossEntropyLoss()
    # Weighted loss function definition
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.02)  # 更强L2

    # Train
    epochs = 40
    model, best_val_acc = train_model(
        model, train_loader, val_loader, criterion, optimizer,
        epochs=epochs, device=device, early_stop_patience=8
    )

    # Test
    test_accuracy = test_model(model, test_loader, device)
    print(f"Test Accuracy: {test_accuracy:.4f}")

Train class distribution: {3: 286, 0: 1798, 1: 273, 2: 227, 7: 258, 8: 253, 6: 294, 4: 236, 5: 225, 9: 94, 10: 89, 11: 20}
Test class distribution: {0: 593, 1: 104, 2: 105, 4: 103, 5: 96}
Class weights: tensor([ 0.1878,  1.2372,  1.4879,  1.1809,  1.4311,  1.5011,  1.1488,  1.3091,
         1.3350,  3.5931,  3.7949, 16.8875])
Epoch 1/40, Loss: 1.9452, Val Accuracy: 0.4414
Epoch 2/40, Loss: 1.0318, Val Accuracy: 0.5265
Epoch 3/40, Loss: 0.7755, Val Accuracy: 0.8594
Epoch 4/40, Loss: 0.6765, Val Accuracy: 0.8496
Epoch 5/40, Loss: 0.5830, Val Accuracy: 0.7719
Epoch 6/40, Loss: 0.5401, Val Accuracy: 0.9075
Epoch 7/40, Loss: 0.5112, Val Accuracy: 0.8730
Epoch 8/40, Loss: 0.4512, Val Accuracy: 0.8619
Epoch 9/40, Loss: 0.4350, Val Accuracy: 0.8952
Epoch 10/40, Loss: 0.4616, Val Accuracy: 0.8718
Epoch 11/40, Loss: 0.4862, Val Accuracy: 0.9026
Epoch 12/40, Loss: 0.4168, Val Accuracy: 0.8705
Epoch 13/40, Loss: 0.4377, Val Accuracy: 0.8705
Epoch 14/40, Loss: 0.4126, Val Accuracy: 0.8866
Epoch 15/

## 5. balanced_fcn_full

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample
import time

# Dataset Class
class BrakeDataset(Dataset):
    def __init__(self, data, labels):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# z-score standardize
def zscore_normalize(train, test):
    train_2d = train.reshape(-1, train.shape[-1])
    mean = np.mean(train_2d, axis=0)
    std = np.std(train_2d, axis=0)
    train_norm = (train - mean) / (std + 1e-8)
    test_norm = (test - mean) / (std + 1e-8)
    return train_norm, test_norm

# Class balance check
def print_class_distribution(labels, name):
    counter = Counter(labels)
    print(f"{name} class distribution: {dict(counter)}")

# upsampling minority classes
def upsample_minority(x, y):
    data = list(zip(x, y))
    class_samples = {c: [d for d in data if d[1]==c] for c in set(y)}
    max_count = max(len(v) for v in class_samples.values())
    new_data = []
    for c, samples in class_samples.items():
        if len(samples) < max_count:
            upsampled = resample(samples, n_samples=max_count, replace=True, random_state=0)
            new_data.extend(upsampled)
        else:
            new_data.extend(samples)
    new_x, new_y = zip(*new_data)
    return np.array(new_x), np.array(new_y)

# FCN Model
class FCN(nn.Module):
    def __init__(self, input_channels, num_classes, dropout_p=0.5):
        super(FCN, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=8)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5)
        self.conv3 = nn.Conv1d(128, 64, kernel_size=3)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# Early Stopping training
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, early_stop_patience=8):
    model = model.to(device)
    best_val_accuracy = 0.0
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                outputs = model(x_batch)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(y_batch.cpu().numpy())

        val_accuracy = accuracy_score(val_labels, val_preds)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss / len(train_loader):.4f}, Val Accuracy: {val_accuracy:.4f}")

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter > early_stop_patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

    if best_model_state:
        model.load_state_dict(best_model_state)
    return model, best_val_accuracy

# Test
def test_model(model, test_loader, device):
    model = model.to(device)
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs = model(x_batch)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(y_batch.cpu().numpy())
    test_accuracy = accuracy_score(test_labels, test_preds)
    return test_accuracy

if __name__ == "__main__":
    # File paths
    train_path = 'train.pickle'
    test_path = 'test.pickle'

    # Load data
    with open(train_path, 'rb') as f:
        train_data = pickle.load(f)
    with open(test_path, 'rb') as f:
        test_data = pickle.load(f)

    x_train = np.array(train_data['sensor_data'].tolist())
    y_train = np.array(train_data['label'])
    x_test = np.array(test_data['sensor_data'].tolist())
    y_test = np.array(test_data['label'])

    # Keep only categories that exist in both training and testing sets
    train_classes = set(np.unique(y_train))
    test_classes = set(np.unique(y_test))
    common_classes = sorted(list(train_classes & test_classes))
    print("Common classes:", common_classes)

    train_mask = np.array([label in common_classes for label in y_train])
    test_mask = np.array([label in common_classes for label in y_test])
    x_train, y_train = x_train[train_mask], y_train[train_mask]
    x_test, y_test = x_test[test_mask], y_test[test_mask]

    print_class_distribution(y_train, "Train")
    print_class_distribution(y_test, "Test")

    # Re-encode categories (ensure labels start from 0 and are continuous)
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    le.fit(common_classes)
    y_train = le.transform(y_train)
    y_test = le.transform(y_test)

    # Data augmentation (upsampling minority classes)
    x_train, y_train = upsample_minority(x_train, y_train)
    print_class_distribution(y_train, "Train (upsampled)")

    # z-score standardize
    x_train, x_test = zscore_normalize(x_train, x_test)

    # transfer to (batch, channel, timestep)
    x_train = np.transpose(x_train, (0, 2, 1))
    x_test = np.transpose(x_test, (0, 2, 1))

    # Split train set and validation set
    dataset = BrakeDataset(x_train, y_train)
    test_dataset = BrakeDataset(x_test, y_test)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

    # DataLoader
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # Model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_channels = x_train.shape[1]
    num_classes = len(common_classes)
    model = FCN(input_channels=input_channels, num_classes=num_classes, dropout_p=0.5)

    # compute class weights
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    print("Class weights:", class_weights)

    # Loss Function and Optimizer
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.02)  # 更强L2

    # Train
    epochs = 40
    model, best_val_acc = train_model(
        model, train_loader, val_loader, criterion, optimizer,
        epochs=epochs, device=device, early_stop_patience=8
    )

    # Test
    test_accuracy = test_model(model, test_loader, device)
    print(f"Test Accuracy: {test_accuracy:.4f}")

Common classes: [0, 1, 2, 4, 5]
Train class distribution: {0: 1798, 1: 273, 2: 227, 4: 236, 5: 225}
Test class distribution: {0: 593, 1: 104, 2: 105, 4: 103, 5: 96}
Train (upsampled) class distribution: {0: 1798, 1: 1798, 2: 1798, 3: 1798, 4: 1798}
Class weights: tensor([1., 1., 1., 1., 1.])
Epoch 1/40, Loss: 0.4284, Val Accuracy: 0.9761
Epoch 2/40, Loss: 0.1542, Val Accuracy: 0.9816
Epoch 3/40, Loss: 0.1401, Val Accuracy: 0.9811
Epoch 4/40, Loss: 0.1277, Val Accuracy: 0.9872
Epoch 5/40, Loss: 0.1198, Val Accuracy: 0.9872
Epoch 6/40, Loss: 0.1147, Val Accuracy: 0.9822
Epoch 7/40, Loss: 0.1173, Val Accuracy: 0.9839
Epoch 8/40, Loss: 0.1168, Val Accuracy: 0.9872
Epoch 9/40, Loss: 0.1121, Val Accuracy: 0.9905
Epoch 10/40, Loss: 0.1077, Val Accuracy: 0.9900
Epoch 11/40, Loss: 0.1032, Val Accuracy: 0.9900
Epoch 12/40, Loss: 0.1050, Val Accuracy: 0.9900
Epoch 13/40, Loss: 0.1047, Val Accuracy: 0.9872
Epoch 14/40, Loss: 0.1014, Val Accuracy: 0.9883
Epoch 15/40, Loss: 0.1014, Val Accuracy: 0.9

## train_with_gan

In [3]:
import pickle
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# ==== 1. load original data ====
with open('train.pickle', 'rb') as f:
    train_data = pickle.load(f)
x_train = np.stack(train_data['sensor_data'])
y_train = np.array(train_data['label'])

# ==== 2. load GAN-generated data ====
with open('generated_data.pickle', 'rb') as f:
    gan_data = pickle.load(f)
x_gan = np.array([item['sensor_data'] for item in gan_data])
y_gan = np.array([item['label'] for item in gan_data])

# ==== 3. load test set ====
with open('test.pickle', 'rb') as f:
    test_data = pickle.load(f)
x_test = np.stack(test_data['sensor_data'])
y_test = np.array(test_data['label'])

# ==== 4. only keep categories which exist in test set ====
test_classes = set(np.unique(y_test))
mask_train = np.array([y in test_classes for y in y_train])
mask_gan = np.array([y in test_classes for y in y_gan])

x_train = x_train[mask_train]
y_train = y_train[mask_train]
x_gan = x_gan[mask_gan]
y_gan = y_gan[mask_gan]

# LabelEncoder only encodes the categories involved in the test set
all_classes = sorted(list(test_classes))
le = LabelEncoder()
le.fit(all_classes)
y_train = le.transform(y_train)
y_gan = le.transform(y_gan)
y_test = le.transform(y_test)

def print_label_stats(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"{name} label dist:", dict(zip(unique, counts)))
print_label_stats("Train", y_train)
print_label_stats("Test", y_test)
print_label_stats("GAN", y_gan)

# ==== 5. Merge GAN data ====
x_train = np.concatenate([x_train, x_gan], axis=0)
y_train = np.concatenate([y_train, y_gan], axis=0)

# ==== 6. Upsampling to achieve class balance ====
num_classes = len(all_classes)
max_count = max([np.sum(y_train == i) for i in range(num_classes)])
x_train_balanced = []
y_train_balanced = []
for i in range(num_classes):
    class_x = x_train[y_train == i]
    class_y = y_train[y_train == i]
    if len(class_x) == 0:
        continue  # Theoretically, it should not occur.
    class_x_upsampled, class_y_upsampled = resample(
        class_x, class_y, replace=True, n_samples=max_count, random_state=42
    )
    x_train_balanced.append(class_x_upsampled)
    y_train_balanced.append(class_y_upsampled)
x_train = np.concatenate(x_train_balanced, axis=0)
y_train = np.concatenate(y_train_balanced, axis=0)

# ==== 7. Standardize the data ====
mean = x_train.mean(axis=(0, 1), keepdims=True)
std = x_train.std(axis=(0, 1), keepdims=True)
x_train = (x_train - mean) / (std + 1e-8)
x_test = (x_test - mean) / (std + 1e-8)

# ==== 8. Split into training/validation sets ====
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# ==== 9. One-hot encode the labels ====
y_train_cat = np.eye(num_classes)[y_train]
y_val_cat = np.eye(num_classes)[y_val]
y_test_cat = np.eye(num_classes)[y_test]

# ==== 10. Reshape Dense inputs ====
x_train = x_train.reshape(x_train.shape[0], -1)
x_val = x_val.reshape(x_val.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

# Convert data to PyTorch tensors
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train_cat = torch.tensor(y_train_cat, dtype=torch.float32)
x_val = torch.tensor(x_val, dtype=torch.float32)
y_val_cat = torch.tensor(y_val_cat, dtype=torch.float32)
x_test = torch.tensor(x_test, dtype=torch.float32)
y_test_cat = torch.tensor(y_test_cat, dtype=torch.float32)

# ==== 11. Build and train the model ====
class MLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.dropout1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = torch.softmax(self.fc3(x), dim=1)
        return x

model = MLP(input_size=x_train.shape[1], num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# DataLoader for batching
train_dataset = TensorDataset(x_train, torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(x_val, torch.tensor(y_val, dtype=torch.long))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128)

# Training loop
epochs = 30
for epoch in range(epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_acc = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            val_acc += (outputs.argmax(dim=1) == batch_y).float().mean().item()
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    print(f"Epoch {epoch+1}/{epochs} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

# ==== 12. Evaluate accuracy on test set ====
model.eval()
test_outputs = model(x_test)
test_acc = (test_outputs.argmax(dim=1) == torch.tensor(y_test)).float().mean().item()
print(f"Test accuracy: {test_acc:.4f}")

# ==== 13. Check for duplicates between training and test sets ====
print(np.intersect1d(x_train.numpy().reshape(x_train.shape[0], -1), x_test.numpy().reshape(x_test.shape[0], -1)).shape)
print("Train label dist:", np.bincount(y_train))
print("Test label dist:", np.bincount(y_test))

Train label dist: {0: 1798, 1: 273, 2: 227, 3: 236, 4: 225}
Test label dist: {0: 593, 1: 104, 2: 105, 3: 103, 4: 96}
GAN label dist: {0: 4053}
Epoch 1/30 - Val Loss: 0.9661, Val Acc: 0.9388
Epoch 2/30 - Val Loss: 0.9649, Val Acc: 0.9398
Epoch 3/30 - Val Loss: 0.9595, Val Acc: 0.9457
Epoch 4/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 5/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 6/30 - Val Loss: 0.9572, Val Acc: 0.9476
Epoch 7/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 8/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 9/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 10/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 11/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 12/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 13/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 14/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 15/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 16/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 17/30 - Val Loss: 0.9571, Val Acc: 0.9478
Epoch 18/30 - Val Loss: 0.9571, Val Acc: 0.9478
Ep

In [8]:

torch.save(model, 'model.pth')